In [ ]:
from datetime import datetime

from numpy.typing import NDArray
import numpy as np

import clustering
from common import grid_search_no_clusters, cross_validate

In [ ]:
def get_clusters(
    adj_matrix: NDArray[np.float64 | np.int32],
    hyperparameters: dict,
    seed: int = 90,
):
    graph = clustering._adjacency_matrix_to_nxgraph(
        adj_matrix, use_disconnected_edges=False
    )
    hyperparameters = {**hyperparameters, "seed": seed}
    clusters = clustering.chinese_whispers_clustering(graph, **hyperparameters)

    return np.array(clustering._convert_graph_cluster_list_set_to_list(graph, clusters))

In [ ]:
def generate_hyperparameters_for_cw():
    return [{"weighting": w} for w in ["top", "lin", "log"]]

In [ ]:
method = "cw"
dataset = "dwug_es"
model = "deepmistake"
path_to_gold_data = "./gold-data-es.csv" 
path_to_data= f"./deepmistake_model_es.csv"

In [ ]:
metadata = {
    "method": method,
    "dataset": dataset,
    "path_to_data": path_to_data,
    "path_to_gold_data": path_to_gold_data,
    "fill_diagonal": True,
    "normalize": True,
    "model": model,
    "use_threshold": True,
}

In [ ]:
start_time = datetime.now()

# grid_search_no_clusters(
#     get_clusters,
#     generate_hyperparameters_for_wsbm(),
#     metadata=metadata,
# )

print(f"Elapsed time {datetime.now() - start_time}")

## Cross-validation experiements

In [ ]:
gold_dir = "./dwug_es_cleaned/clusters"

In [ ]:
start_time = datetime.now()

cv_summary = cross_validate(
    get_clusters,
    generate_hyperparameters_for_cw(),
    metadata=metadata,
    gold_dir=gold_dir,
    k=5,
)

print(f"  Elapsed time: {datetime.now() - start_time}")

In [ ]:
print(f"\nProtocol 1 (ARI driven): ")
print(f"  avg test ARI: {cv_summary['protocol_ari']['avg_test_ari']:.4f}")
print(f"  avg test LSCD: {cv_summary['protocol_ari']['avg_test_lscd']:.4f}")
print(f"\nProtocol 2 (LSCD Driven):")
print(f"  avg test LSCD: {cv_summary['protocol_lscd']['avg_test_lscd']:.4f}")
print(f"  avg test ARI: {cv_summary['protocol_lscd']['avg_test_ari']:.4f}")